In [74]:
import math
import warnings
from pathlib import Path
from plotly import graph_objects as go
import numpy as np
import numpy.typing as npt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
import random
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")

In [75]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Makes cuDNN deterministic (slight perf cost, irrelevant on MPS)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [76]:
DATA_DIR = Path("../data")
CHECKPOINT_DIR = Path("./checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
DEVICE = (
    torch.device("cuda") if torch.cuda.is_available() else 
    torch.device("mps") if torch.backends.mps.is_available() else
    torch.device("cpu")
)

## Data Loading

In [77]:
class WindowDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        feature_cols: list[str],
        seq_len: int = 20,
        target_col: str = "target",
    ):
        self.seq_len = seq_len
        self.samples: list[tuple[npt.NDArray[np.float32], np.float32]] = []

        for ticker, group in df.groupby("ticker"):
            group = group.sort_index()
            X = group[feature_cols].values.astype(np.float32)
            y = group[target_col].values.astype(np.float32)

            mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
            X = X[mask]
            y = y[mask]

            for i in range(seq_len, len(group)):
                window = X[i - seq_len : i]
                target = y[i]
                self.samples.append((window, target))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.from_numpy(x), torch.tensor(y)

In [78]:
def make_loaders(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
    seq_len: int = 20,
    batch_size: int = 64,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    kw = dict(feature_cols=feature_cols, seq_len=seq_len)
    train_ds = WindowDataset(train_df, **kw)
    val_ds = WindowDataset(val_df, **kw)
    test_ds = WindowDataset(test_df, **kw)

    loader_kw = dict(batch_size=batch_size, num_workers=0)  # num_workers=0 on MPS
    return (
        DataLoader(train_ds, shuffle=True, **loader_kw),
        DataLoader(val_ds, shuffle=False, **loader_kw),
        DataLoader(test_ds, shuffle=False, **loader_kw),
    )

## Model
Differnce from baseline GRU: Skip layer + add depth in head

In [79]:
class DirectionalHuberLoss(nn.Module):
    """Huber + bonus when sign(pred) == sign(target)."""
    def __init__(self, delta=1.0, dir_weight=0.3):
        super().__init__()
        self.huber = nn.HuberLoss(delta=delta, reduction="none")
        self.dir_weight = dir_weight

    def forward(self, preds, targets):
        huber = self.huber(preds, targets).mean()
        # Reward correct direction, penalize wrong direction
        sign_match = (preds.sign() == targets.sign()).float()
        dir_loss = 1.0 - sign_match.mean()
        return huber + self.dir_weight * dir_loss

In [80]:
class AdditiveAttention(nn.Module):
    """
    Bahdanau-style attention over the time dimension.

    Given hidden states H  (B, T, H), produces a context vector (B, H)
    that is a weighted sum of all timesteps.

    Score:  e_t = v · tanh(W · h_t)      (learned scalar per timestep)
    Weight: α   = softmax(e)              (over T)
    Output: c   = Σ α_t * h_t
    """

    def __init__(self, hidden_size: int):
        super().__init__()
        self.W = nn.Linear(hidden_size, hidden_size, bias=True)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden_states: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # hidden_states: (B, T, H)
        scores = self.v(torch.tanh(self.W(hidden_states)))  # (B, T, 1)
        weights = torch.softmax(scores, dim=1)  # (B, T, 1)
        context = (weights * hidden_states).sum(dim=1)  # (B, H)
        return context, weights.squeeze(-1)  # context, attn_map

In [81]:
class DirectionalBCELoss(nn.Module):
    """Treat return prediction as a classification: positive vs negative."""
    def forward(self, preds, targets):
        # Convert targets to binary labels: 1 if positive, 0 if negative
        labels = (targets > 0).float()
        # Sigmoid on raw predictions → probability of "up"
        return F.binary_cross_entropy_with_logits(preds, labels)

In [82]:
class SoftDirectionalHuberLoss(nn.Module):
    """Huber + differentiable directional penalty."""

    def __init__(self, delta=1.01, dir_weight=0.3, sharpness=10.0):
        super().__init__()
        self.huber = nn.HuberLoss(delta=delta, reduction="none")
        self.dir_weight = dir_weight
        self.sharpness = sharpness  # higher = closer to hard sign

    def forward(self, preds, targets):
        huber = self.huber(preds, targets).mean()

        # Approach 1: soft sign agreement via tanh
        # Product is positive when same sign, negative when opposite
        agreement = torch.tanh(preds * self.sharpness) * torch.tanh(targets * self.sharpness)
        dir_loss = -agreement.mean()  # minimize → maximize agreement

        return huber + self.dir_weight * dir_loss

class TemporalCNNHead(nn.Module):
    def __init__(self, hidden_size: int, out_size: int, 
                 channels: int = 64, kernel_sizes: list = [3, 5]):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(hidden_size, channels, k, padding=k//2),
                nn.GELU(),
                nn.BatchNorm1d(channels),
            )
            for k in kernel_sizes
        ])
        # each kernel produces `channels` features → concat
        combined = channels * len(kernel_sizes)
        self.fc = nn.Sequential(
            nn.Linear(combined, out_size),
        )

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        # hidden_states: (B, T, H) → conv expects (B, H, T)
        x = hidden_states.transpose(1, 2)
        
        # apply each kernel, global-average-pool over time
        pooled = [conv(x).mean(dim=-1) for conv in self.convs]  # [(B, C), ...]
        out = torch.cat(pooled, dim=1)  # (B, C*n_kernels)
        return self.fc(out)             # (B, out_size)

In [83]:
class GRUWithAttentionTCN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=2,
                 dropout=0.35, bypass_window=10,
                 tcn_channels=64, tcn_kernels=[3, 5]):
        super().__init__()
        self.bypass_window = bypass_window
        
        self.gru = nn.GRU(input_size, hidden_size, num_layers,
                          batch_first=True,
                          dropout=dropout if num_layers > 1 else 0.0)
        self.attention = AdditiveAttention(hidden_size)
        
        self.tcn_head = TemporalCNNHead(
            hidden_size, 
            out_size=hidden_size,   # project to same size for concat
            channels=tcn_channels, 
            kernel_sizes=tcn_kernels
        )
        
        # final: attention context + skip + tcn output
        self.head = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
        )

        self.layer_norm = nn.BatchNorm1d(hidden_size)


    def forward(self, x):
        hidden_states, _ = self.gru(x)
        context, attn = self.attention(hidden_states)
        tcn_out = self.tcn_head(hidden_states)
        tcn_out = self.layer_norm(tcn_out)
        combined = torch.cat([context, tcn_out], dim=-1)
        return self.head(combined).squeeze(-1), attn

## Model torchure

In [84]:
def train_one_epoch(
    model: GRUWithAttention,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    clip_grad: float = 1.0,
) -> float:
    model.train()
    total_loss = 0.0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        preds, _ = model(x)
        loss = criterion(preds, y)
        loss.backward()

        if clip_grad > 0:
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)

        optimizer.step()
        total_loss += loss.item() * len(y)

    return total_loss / len(loader.dataset)

In [85]:
@torch.no_grad()
def evaluate(
    model: GRUWithAttention,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> dict:
    """
    Returns a dict with loss, MAE, and directional accuracy.

    Directional accuracy: fraction of predictions where sign(pred) == sign(true).
    This is the headline metric for a return-prediction model –
    a coin-flip baseline is 0.50.
    """
    model.eval()
    all_preds, all_targets = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        preds, _ = model(x)
        all_preds.append(preds.cpu())
        all_targets.append(y.cpu())

    preds = torch.cat(all_preds)
    targets = torch.cat(all_targets)

    loss = criterion(preds, targets).item()
    mae = (preds - targets).abs().mean().item()
    dir_acc = (preds.sign() == targets.sign()).float().mean().item()

    return {"loss": loss, "mae": mae, "dir_acc": dir_acc}

In [86]:
def train(
    model:        GRUWithAttention,
    train_loader: DataLoader,
    val_loader:   DataLoader,
    device:       torch.device,
    lr:           float = 1e-4,
    epochs:       int   = 50,
    patience:     int   = 10,
    huber_delta:  float = 1.0,
) ->GRUWithAttention:
    """
    Full training loop with:
     - Huber loss (delta=1.0 on z-scored targets is well calibrated)
     - AdamW + cosine LR schedule
     - Early stopping on val loss
     - W&B logging
    """
    criterion = SoftDirectionalHuberLoss(delta=huber_delta)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_loss = float("inf")
    patience_ctr  = 0
    best_state    = None

    logs = []

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        logs.append({
            "epoch":          epoch,
            "train/loss":     train_loss,
            "val/loss":       val_metrics["loss"],
            "val/mae":        val_metrics["mae"],
            "val/dir_acc":    val_metrics["dir_acc"],
                "lr":             scheduler.get_last_lr()[0],
            })

        print(
            f"Epoch {epoch:3d} | "
            f"train_loss={train_loss:.4f}  "
            f"val_loss={val_metrics['loss']:.4f}  "
            f"val_dir_acc={val_metrics['dir_acc']:.3f}"
        )

        # Early stopping
        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr  = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"Early stop at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    return model


In [87]:
train_df = pd.read_csv("data/train_features.csv", index_col=0, parse_dates=True)
val_df   = pd.read_csv("data/val_features.csv",   index_col=0, parse_dates=True)
test_df  = pd.read_csv("data/test_features.csv",  index_col=0, parse_dates=True)


# Combine train + val for CV; keep test held out as final evaluation
cv_df   = pd.concat([train_df, val_df]).sort_index()
cv_df.sort_index(inplace=True)
print(f"CV pool:  {len(cv_df):,} rows  |  "
      f"{cv_df.index.min().date()} → {cv_df.index.max().date()}")
print(f"Test set: {len(test_df):,} rows (held out)")

feature_cols = [
    # Regime / trend (strongest single predictor, r=-0.067)
    "bull_regime",
    
    # Momentum (r=+0.045, captures "where in Bollinger Band")
    "z_bb_pct_b",
    
    # Volume anomaly (r=+0.041, orthogonal to price features)
    "z_volume_z20",
    
    # Yesterday's return (r=-0.021, short-term mean reversion signal)
    "z_log_close_return_1",
    
    # Volatility (r=+0.031, "how wild was today")
    "z_range",
    
    # Autocorrelation (r=+0.024, trending vs mean-reverting regime)
    "z_ret_autocorr",
    
    # Calendar (weak but orthogonal to everything else)
    "dow_cos",
    
    # Vol regime (binary, orthogonal to z_range which measures magnitude)
    "high_vol_regime",
]
print(f"Features ({len(feature_cols)}): {feature_cols[:5]} ...")

CV pool:  6,269 rows  |  2015-04-03 → 2025-05-28
Test set: 570 rows (held out)
Features (8): ['bull_regime', 'z_bb_pct_b', 'z_volume_z20', 'z_log_close_return_1', 'z_range'] ...


In [88]:
class DirectionalHuberLoss(nn.Module):
    """Huber + bonus when sign(pred) == sign(target)."""
    def __init__(self, delta=1.0, dir_weight=0.3):
        super().__init__()
        self.huber = nn.HuberLoss(delta=delta, reduction="none")
        self.dir_weight = dir_weight

    def forward(self, preds, targets):
        huber = self.huber(preds, targets).mean()
        # Reward correct direction, penalize wrong direction
        sign_match = (preds.sign() == targets.sign()).float()
        dir_loss = 1.0 - sign_match.mean()
        return huber + self.dir_weight * dir_loss
    
class SoftDirectionalHuberLoss(nn.Module):
    """Huber + differentiable directional penalty."""

    def __init__(self, delta=1.01, dir_weight=0.3, sharpness=10.0):
        super().__init__()
        self.huber = nn.HuberLoss(delta=delta, reduction="none")
        self.dir_weight = dir_weight
        self.sharpness = sharpness  # higher = closer to hard sign

    def forward(self, preds, targets):
        huber = self.huber(preds, targets).mean()

        # Approach 1: soft sign agreement via tanh
        # Product is positive when same sign, negative when opposite
        agreement = torch.tanh(preds * self.sharpness) * torch.tanh(targets * self.sharpness)
        dir_loss = -agreement.mean()  # minimize → maximize agreement

        return huber + self.dir_weight * dir_loss


class CorrelationHuberLoss(nn.Module):
    """Huber + simple correlation-style penalty (simplest version)."""

    def __init__(self, delta=1.01, dir_weight=0.3 ):
        super().__init__()
        self.huber = nn.HuberLoss(delta=delta, reduction="none")
        self.dir_weight = dir_weight

    def forward(self, preds, targets):
        huber = self.huber(preds, targets).mean()

        # Approach 2: negative mean product
        # pred * target > 0 when same sign → we want to maximize it
        dir_loss = -(preds * targets).mean()

        return huber + self.dir_weight * dir_loss

In [89]:
from timeseries_cv import cross_validate, summarize_cv, compare_strategies
from timeseries_cv import expanding_window_folds, sliding_window_folds

criterion = nn.HuberLoss(delta=1.01) # for quick sanity check; not directional

print("EXPANDING:")
for f in expanding_window_folds(cv_df, n_folds=5, min_train_frac=0.40, val_frac=0.10):
    print(f"  Fold {f.fold_idx}: train {f.train_start.date()} → {f.train_end.date()} "
          f"| val {f.val_start.date()} → {f.val_end.date()}")
    targets  = cv_df.loc[f.val_start : f.val_end, "target"]
    loss = criterion(
        torch.zeros_like(torch.from_numpy(targets.values)),  # dummy preds
        torch.from_numpy(targets.values)
    ).item()
    print(f"    Huber loss on val set (dummy preds): {loss:.4f}")

print("\nSLIDING:")
for f in sliding_window_folds(cv_df, n_folds=5, train_frac=0.40, val_frac=0.10):
    print(f"  Fold {f.fold_idx}: train {f.train_start.date()} → {f.train_end.date()} "
          f"| val {f.val_start.date()} → {f.val_end.date()}")

EXPANDING:
  Fold 0: train 2015-04-03 → 2019-04-24 | val 2019-04-25 → 2020-04-28
    Huber loss on val set (dummy preds): 0.3973
  Fold 1: train 2015-04-03 → 2020-07-31 | val 2020-08-01 → 2021-08-05
    Huber loss on val set (dummy preds): 0.4410
  Fold 2: train 2015-04-03 → 2021-11-07 | val 2021-11-08 → 2022-11-12
    Huber loss on val set (dummy preds): 0.3917
  Fold 3: train 2015-04-03 → 2023-02-14 | val 2023-02-15 → 2024-02-19
    Huber loss on val set (dummy preds): 0.3693
  Fold 4: train 2015-04-03 → 2024-05-23 | val 2024-05-24 → 2025-05-28
    Huber loss on val set (dummy preds): 0.3768

SLIDING:
  Fold 0: train 2015-04-03 → 2019-04-24 | val 2019-04-25 → 2020-04-28
  Fold 1: train 2016-07-10 → 2020-07-31 | val 2020-08-01 → 2021-08-05
  Fold 2: train 2017-10-17 → 2021-11-07 | val 2021-11-08 → 2022-11-12
  Fold 3: train 2019-01-24 → 2023-02-14 | val 2023-02-15 → 2024-02-19
  Fold 4: train 2020-05-02 → 2024-05-23 | val 2024-05-24 → 2025-05-28


In [101]:
SEQ_LEN = 30

def model_factory():
    torch.manual_seed(SEED)
    return GRUWithAttentionTCN(
    input_size=len(feature_cols), hidden_size=64,
    num_layers=2, tcn_channels=64, dropout=0.25, bypass_window=10, tcn_kernels=[5, 7, 10]
).to(DEVICE)

shared_params = dict(
    full_df=cv_df, feature_cols=feature_cols,
    model_factory=model_factory, dataset_cls=WindowDataset,
    n_folds=5, val_frac=0.10,
    seq_len=SEQ_LEN, batch_size=64, lr=2e-4,
    epochs=1000, patience=30, huber_delta=1.01, device=DEVICE,
    criterion=criterion, weight_decay=2e-1, warmup_epochs=5, train_frac=0.4
)


In [91]:

exp_results = cross_validate(**shared_params, strategy="expanding")
exp_df = summarize_cv(exp_results)



  [EXPANDING] Fold 0  |  train 2015-04-03 → 2019-04-24  |  val 2019-04-25 → 2020-04-28
  Epoch   1  train_loss=0.3752  val_loss=0.3886  dir_acc=0.493
  Epoch   2  train_loss=0.3697  val_loss=0.3881  dir_acc=0.493
  Epoch   3  train_loss=0.3696  val_loss=0.3867  dir_acc=0.493
  Epoch   4  train_loss=0.3683  val_loss=0.3838  dir_acc=0.493
  Epoch   5  train_loss=0.3692  val_loss=0.3796  dir_acc=0.518
  Epoch   6  train_loss=0.3644  val_loss=0.3806  dir_acc=0.507
  Epoch   7  train_loss=0.3676  val_loss=0.3790  dir_acc=0.510
  Epoch   8  train_loss=0.3648  val_loss=0.3796  dir_acc=0.507
  Epoch   9  train_loss=0.3657  val_loss=0.3804  dir_acc=0.524
  Epoch  10  train_loss=0.3649  val_loss=0.3803  dir_acc=0.506
  Epoch  11  train_loss=0.3647  val_loss=0.3799  dir_acc=0.515
  Epoch  12  train_loss=0.3635  val_loss=0.3806  dir_acc=0.512
  Epoch  13  train_loss=0.3641  val_loss=0.3809  dir_acc=0.507
  Epoch  14  train_loss=0.3650  val_loss=0.3804  dir_acc=0.519
  Epoch  15  train_loss=0.3639

In [92]:

sli_results = cross_validate(**shared_params, strategy="sliding")
sli_df = summarize_cv(sli_results) 


  [SLIDING] Fold 0  |  train 2015-04-03 → 2019-04-24  |  val 2019-04-25 → 2020-04-28
  Epoch   1  train_loss=0.3752  val_loss=0.3886  dir_acc=0.493
  Epoch   2  train_loss=0.3697  val_loss=0.3881  dir_acc=0.493
  Epoch   3  train_loss=0.3696  val_loss=0.3867  dir_acc=0.493
  Epoch   4  train_loss=0.3683  val_loss=0.3838  dir_acc=0.493
  Epoch   5  train_loss=0.3692  val_loss=0.3796  dir_acc=0.518
  Epoch   6  train_loss=0.3644  val_loss=0.3806  dir_acc=0.507
  Epoch   7  train_loss=0.3676  val_loss=0.3790  dir_acc=0.510
  Epoch   8  train_loss=0.3648  val_loss=0.3796  dir_acc=0.507
  Epoch   9  train_loss=0.3657  val_loss=0.3804  dir_acc=0.524
  Epoch  10  train_loss=0.3649  val_loss=0.3803  dir_acc=0.506
  Epoch  11  train_loss=0.3647  val_loss=0.3799  dir_acc=0.515
  Epoch  12  train_loss=0.3635  val_loss=0.3806  dir_acc=0.512
  Epoch  13  train_loss=0.3641  val_loss=0.3809  dir_acc=0.507
  Epoch  14  train_loss=0.3650  val_loss=0.3804  dir_acc=0.519
  Epoch  15  train_loss=0.3639  

In [93]:
comp_df = compare_strategies(exp_results, sli_results)

comp_df.head()


  STRATEGY COMPARISON
  expanding   |  dir_acc = 0.541 ± 0.009  |  val_loss = 0.3928 ± 0.0309
  sliding     |  dir_acc = 0.527 ± 0.005  |  val_loss = 0.3931 ± 0.0308

  → Expanding wins: model benefits from more history.


,strategy,val_loss_mean,val_loss_std,dir_acc_mean,dir_acc_std,n_folds,best_fold_acc,worst_fold_acc
0,expanding,0.392820,0.030911,0.541176,0.009439,5,0.558824,0.532353
1,sliding,0.393051,0.030799,0.527059,0.005471,5,0.535294,0.519118


In [94]:
import plotly.graph_objects as go

fold_labels = [f"Fold {r.fold_idx}" for r in exp_results]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=fold_labels,
    y=[r.val_dir_acc for r in exp_results],
    name="Expanding", marker_color="#636efa", opacity=0.8,
))
fig.add_trace(go.Bar(
    x=fold_labels,
    y=[r.val_dir_acc for r in sli_results],
    name="Sliding", marker_color="#ef553b", opacity=0.8,
))
fig.add_hline(y=0.50, line_dash="dash", line_color="white",
              annotation_text="coin flip")
fig.update_layout(
    template="plotly_dark",
    title="Expanding vs Sliding Window — Dir. Accuracy per Fold",
    yaxis_title="Dir. Accuracy",
    yaxis_range=[0.40, 0.65],
    barmode="group",
)
fig.show()

In [95]:
from timeseries_cv import evaluate_holdout
btc_result = evaluate_holdout(
    train_df=cv_df, eval_df=test_df,
    feature_cols=feature_cols,
    model_factory=model_factory,
    dataset_cls=WindowDataset,
    lr=1e-3, device=DEVICE,
)



  HOLDOUT EVALUATION  |  ticker=ALL
  train 2015-04-03 -> 2025-05-28  (6,269 rows)
  eval  2025-05-29 -> 2026-03-09  (570 rows)
  Epoch   1  train_loss=0.3815  eval_loss=0.4190  dir_acc=0.542
  Epoch   2  train_loss=0.3790  eval_loss=0.4170  dir_acc=0.534
  Epoch   3  train_loss=0.3785  eval_loss=0.4198  dir_acc=0.521
  Epoch   4  train_loss=0.3770  eval_loss=0.4251  dir_acc=0.521
  Epoch   5  train_loss=0.3770  eval_loss=0.4181  dir_acc=0.528
  Epoch   6  train_loss=0.3762  eval_loss=0.4180  dir_acc=0.543
  Epoch   7  train_loss=0.3759  eval_loss=0.4231  dir_acc=0.492
  Epoch   8  train_loss=0.3755  eval_loss=0.4190  dir_acc=0.551
  Epoch   9  train_loss=0.3756  eval_loss=0.4204  dir_acc=0.534
  Epoch  10  train_loss=0.3750  eval_loss=0.4195  dir_acc=0.562
  Epoch  11  train_loss=0.3750  eval_loss=0.4178  dir_acc=0.545
  Epoch  12  train_loss=0.3750  eval_loss=0.4187  dir_acc=0.540
  Epoch  13  train_loss=0.3748  eval_loss=0.4190  dir_acc=0.494
  Epoch  14  train_loss=0.3745  eval_lo

In [96]:
import importlib
import timeseries_cv
importlib.reload(timeseries_cv)
from timeseries_cv import evaluate_holdout, EvalResult


In [103]:
btc_result = evaluate_holdout(
    train_df=cv_df, eval_df=test_df,
    feature_cols=feature_cols,
    model_factory=model_factory,
    dataset_cls=WindowDataset,
    lr=1e-3, device=DEVICE,
    ticker=["BTC-USD", "ETH-USD"],
)



  HOLDOUT EVALUATION  |  ticker=BTC-USD, ETH-USD
  train 2015-04-03 -> 2025-05-28  (6,269 rows)
  eval  2025-05-29 -> 2026-03-09  (570 rows)
  Epoch   1  train_loss=0.3815  eval_loss=0.4190  dir_acc=0.542
  Epoch   2  train_loss=0.3790  eval_loss=0.4170  dir_acc=0.534
  Epoch   3  train_loss=0.3785  eval_loss=0.4198  dir_acc=0.521
  Epoch   4  train_loss=0.3770  eval_loss=0.4251  dir_acc=0.521
  Epoch   5  train_loss=0.3770  eval_loss=0.4181  dir_acc=0.528
  Epoch   6  train_loss=0.3762  eval_loss=0.4180  dir_acc=0.543
  Epoch   7  train_loss=0.3759  eval_loss=0.4231  dir_acc=0.492
  Epoch   8  train_loss=0.3755  eval_loss=0.4190  dir_acc=0.551
  Epoch   9  train_loss=0.3756  eval_loss=0.4204  dir_acc=0.534
  Epoch  10  train_loss=0.3750  eval_loss=0.4195  dir_acc=0.562
  Epoch  11  train_loss=0.3750  eval_loss=0.4178  dir_acc=0.545
  Epoch  12  train_loss=0.3750  eval_loss=0.4187  dir_acc=0.540
  Epoch  13  train_loss=0.3748  eval_loss=0.4190  dir_acc=0.494
  Epoch  14  train_loss=0.

In [98]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=btc_result.eval_dates, y=btc_result.eval_targets, name="actual"))
fig.add_trace(go.Scatter(x=btc_result.eval_dates, y=btc_result.eval_preds,   name="predicted"))
fig.show()